In [1]:
# --- Workspace Verification ---
# This cell loads the artifacts from your previous notebook to ensure 
# the training pipeline has all necessary components in memory.

import torch
import pandas as pd
import pickle
import joblib
from pathlib import Path
from torch_geometric.data import Data

# 1. Setup Paths
root = Path("/home/jupyter-1nt23cb058/Capstone")
tensor_dir = root / "data/processed/tensors"
graph_path = root / "data/processed/graph/topology_graph.pt"
scaler_path = root / "data/processed/graph/global_scaler.pkl"

print("Checking artifacts...")

# 2. Load the Graph
# Note: We handled the 'dict' vs 'Data' object issue in the previous notebook.
graph_data = torch.load(graph_path)
if isinstance(graph_data, dict) and 'edge_index' in graph_data:
    # Reconstruct from dictionary if necessary
    edge_index = graph_data['edge_index']
    num_nodes = graph_data.get('num_nodes', edge_index.max().item() + 1)
    graph = Data(edge_index=edge_index, num_nodes=num_nodes)
else:
    graph = graph_data

# 3. Load Tensors
X_train = torch.load(tensor_dir / "X_train.pt")
y_train = torch.load(tensor_dir / "y_train.pt")
node_ids = torch.load(tensor_dir / "node_ids.pt")

# 4. Load Scaler
with open(scaler_path, 'rb') as f:
    scaler = joblib.load(f)

# 5. Device Setup (V100 check)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Output Report ---
print("-" * 30)
print(f"Device:          {device}")
print(f"X_train Shape:   {X_train.shape} (Sequences, Time, Features)")
print(f"y_train Shape:   {y_train.shape}")
print(f"Graph Nodes:     {graph.num_nodes:,}")
print(f"Graph Edges:     {graph.edge_index.shape[1]:,}")
print(f"Target Mean:     {scaler['target']['mean']:.4f}")
print("-" * 30)
print("READY FOR MODEL ARCHITECTURE.")

Checking artifacts...
------------------------------
Device:          cuda
X_train Shape:   torch.Size([1069795, 24, 17]) (Sequences, Time, Features)
y_train Shape:   torch.Size([1069795, 1])
Graph Nodes:     154,902
Graph Edges:     424,550
Target Mean:     39.2206
------------------------------
READY FOR MODEL ARCHITECTURE.


In [3]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

print("Seed set:", SEED)

Seed set: 42


In [4]:
# Optional confidence and source artifacts
# If not found, defaults to confidence=1.0 for all samples.

source_conf_path = tensor_dir / "source_confidence.pt"
label_source_path = tensor_dir / "label_source.pt"

if source_conf_path.exists():
    source_confidence = torch.load(source_conf_path).float()
else:
    source_confidence = torch.ones(X_train.shape[0], dtype=torch.float32)

if label_source_path.exists():
    label_source = torch.load(label_source_path)
else:
    label_source = None

print("source_confidence shape:", tuple(source_confidence.shape))
print("label_source present:", label_source is not None)

source_confidence shape: (1069795,)
label_source present: False


In [5]:
# Temporal split proxy on sequence index: first 80 percent train, last 20 percent val
# This is a trial split. Use strict time windows in full experiment notebook.

n_samples = X_train.shape[0]
split_idx = int(n_samples * 0.80)

idx_train = torch.arange(0, split_idx)
idx_val = torch.arange(split_idx, n_samples)

print("Total samples:", n_samples)
print("Train samples:", idx_train.numel())
print("Val samples:", idx_val.numel())

Total samples: 1069795
Train samples: 855836
Val samples: 213959


In [6]:
class SeqDataset(Dataset):
    def __init__(self, X, y, w):
        self.X = X.float()
        self.y = y.float()
        self.w = w.float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i], self.w[i]


X_tr = X_train[idx_train]
y_tr = y_train[idx_train]
w_tr = source_confidence[idx_train]

X_va = X_train[idx_val]
y_va = y_train[idx_val]
w_va = source_confidence[idx_val]

train_ds = SeqDataset(X_tr, y_tr, w_tr)
val_ds = SeqDataset(X_va, y_va, w_va)

train_loader = DataLoader(
    train_ds,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=1024,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train loader batches:", len(train_loader))
print("Val loader batches:", len(val_loader))

Train loader batches: 1672
Val loader batches: 209


In [12]:
# Replacement Cell: model + optimizer + AMP
# Fix for older PyTorch versions where ReduceLROnPlateau has no verbose= argument

import copy
import torch
import torch.nn as nn


class GRUTrialRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=1, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_size=in_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        out, _ = self.gru(x)
        h_last = out[:, -1, :]
        return self.head(h_last).squeeze(-1)


in_dim = X_train.shape[-1]

model = GRUTrialRegressor(
    in_dim=in_dim,
    hidden_dim=64,
    num_layers=1,
    dropout=0.3
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=5e-4
)

# Compatible across older/newer PyTorch builds
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-5
)

amp_enabled = (device.type == "cuda")

if amp_enabled:
    scaler_amp = torch.amp.GradScaler("cuda")
else:
    scaler_amp = torch.amp.GradScaler(enabled=False)

print("Model initialized with in_dim =", in_dim)
print("PyTorch version:", torch.__version__)

Model initialized with in_dim = 17
PyTorch version: 2.8.0+cu128


In [13]:
# Replacement Cell: weighted loss + early stopping training loop

def weighted_mse(pred, target, weight):
    pred = pred.view(-1)
    target = target.view(-1)
    weight = weight.view(-1).clamp(min=0.05, max=1.0)

    loss = ((pred - target) ** 2) * weight
    return loss.sum() / (weight.sum() + 1e-8)


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss = 0.0
    total_count = 0

    for xb, yb, wb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True).view(-1)
        wb = wb.to(device, non_blocking=True).view(-1)

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            pred = model(xb)
            loss = weighted_mse(pred, yb, wb)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()

        bs = xb.shape[0]
        total_loss += float(loss.detach().cpu()) * bs
        total_count += bs

    return total_loss / max(total_count, 1)


MAX_EPOCHS = 15
PATIENCE = 3

best_val = float("inf")
best_epoch = -1
best_state = None
no_improve = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss = run_epoch(model, val_loader, optimizer=None)

    scheduler.step(val_loss)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": optimizer.param_groups[0]["lr"]
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={val_loss:.6f} | "
        f"lr={row['lr']:.2e}"
    )

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch}, "
            f"best_val={best_val:.6f}"
        )
        break

if best_state is not None:
    model.load_state_dict(best_state)

print({
    "best_epoch": best_epoch,
    "best_val_loss": float(best_val)
})

Epoch 01 | train_loss=0.471889 | val_loss=0.571018 | lr=1.00e-03
Epoch 02 | train_loss=0.195925 | val_loss=0.728193 | lr=1.00e-03
Epoch 03 | train_loss=0.161395 | val_loss=0.690235 | lr=5.00e-04
Epoch 04 | train_loss=0.136329 | val_loss=0.719131 | lr=5.00e-04
Early stopping at epoch 4. Best epoch: 1, best_val=0.571018
{'best_epoch': 1, 'best_val_loss': 0.5710176148546725}


In [14]:
# Replacement Cell: final validation metrics using best checkpoint

import math
import numpy as np

model.eval()

preds = []
targets = []

with torch.no_grad():
    for xb, yb, wb in val_loader:
        xb = xb.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            pred = model(xb)

        preds.append(pred.detach().cpu())
        targets.append(yb.view(-1).cpu())

pred = torch.cat(preds).numpy()
true = torch.cat(targets).numpy()

mae = float(np.mean(np.abs(pred - true)))
rmse = float(math.sqrt(np.mean((pred - true) ** 2)))

print("-" * 44)
print("Best-Checkpoint Validation Metrics")
print("best_epoch   :", best_epoch)
print("best_val_loss:", round(float(best_val), 6))
print("MAE          :", round(mae, 4))
print("RMSE         :", round(rmse, 4))
print("-" * 44)

--------------------------------------------
Best-Checkpoint Validation Metrics
best_epoch   : 1
best_val_loss: 0.571018
MAE          : 0.5501
RMSE         : 0.7557
--------------------------------------------


In [10]:
def lambda_phys_schedule(epoch, warmup_epochs=8, max_lambda=0.25):
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda


for e in range(12):
    if e % 2 == 0:
        print({
            "epoch": e,
            "lambda_phys": round(lambda_phys_schedule(e), 4)
        })

{'epoch': 0, 'lambda_phys': 0.0312}
{'epoch': 2, 'lambda_phys': 0.0938}
{'epoch': 4, 'lambda_phys': 0.1562}
{'epoch': 6, 'lambda_phys': 0.2188}
{'epoch': 8, 'lambda_phys': 0.25}
{'epoch': 10, 'lambda_phys': 0.25}
